# 🏭 Physics-Informed Data Generator
## Synthetic Stellar Streams & Galactic Cirrus Pipeline

**Author:** Ibon Garcia Gomez  
**Context:** Master's Thesis (TFM) Research  
**Output:** 20,000 Multi-band FITS images (SDSS g/r + WISE W1/W2)  
**Strategy:** Hard Example Mining (40% Edge Cases)

---

### 🎯 Objective
To train a robust Deep Learning model for Low-Surface Brightness (LSB) detection, we need a massive dataset with ground truth masks. Since real labeled data for stellar streams is scarce, this notebook generates synthetic samples by injecting procedurally generated structures into real background fields.

### ⚙️ Methodology
1.  **Real Backgrounds:** Downloads random fields from SDSS and WISE surveys via HiPS (Aladin Lite).
2.  **Procedural Structures:**
    * **Stellar Streams:** Modeled using quadratic Bézier curves with stochastic $1/f^\beta$ fractal noise textures sampled from **$\beta \in [1.5, 2.5]$**.
    * **Galactic Cirrus:** Generated using Brownian motion fields with **$\beta \in [2.5, 3.5]$** to mimic the variable density of interstellar dust.
3.  **Hard Mining Strategy:**
    * **Normal Set (12,000):** Standard SNRs and random overlaps.
    * **Hard Set (8,000):** Forced high-contrast overlaps (cirrus masking streams) and critically low SNR streams to force the model to learn edge cases.

In [1]:
import os
import random
import shutil
import numpy as np
import requests
import time
from io import BytesIO
from astropy.io import fits
from skimage.draw import bezier_curve
from scipy.ndimage import gaussian_filter
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
class Config:
    # Output Directories
    BASE_OUTPUT = '/kaggle/working/synthetic_dataset'
    IMG_PATH = os.path.join(BASE_OUTPUT, 'images')
    MASK_PATH = os.path.join(BASE_OUTPUT, 'masks')
    
    # Dataset Balance
    N_NORMAL = 12000     # Standard samples
    N_HARD = 8000        # Edge cases (Hard Mining)
    TOTAL = N_NORMAL + N_HARD
    
    # Image Parameters
    IMG_SIZE = 128
    SURVEYS = [
        'CDS/P/SDSS9/g',   # Visible Green
        'CDS/P/SDSS9/r',   # Visible Red (Stream dominant)
        'CDS/P/allWISE/W1',# IR 3.4um (Cirrus dominant)
        'CDS/P/allWISE/W2' # IR 4.6um
    ]
    
    # Parallel Processing
    MAX_WORKERS = 8      # Optimized for Kaggle CPU
    TIMEOUT = 15         # Seconds per HTTP request

# Setup Directories
if os.path.exists(Config.BASE_OUTPUT):
    shutil.rmtree(Config.BASE_OUTPUT)
os.makedirs(Config.IMG_PATH, exist_ok=True)
os.makedirs(Config.MASK_PATH, exist_ok=True)

print(f"🚀 Generator Configured: {Config.TOTAL} images target.")

🚀 Generator Configured: 20000 images target.


## 📐 Fractal Noise Engine — Astrophysical Justification

The core realism of this generator relies on **$1/f^\beta$ colored noise** (also known as fractional Brownian motion), which is the standard model for the power spectrum of interstellar medium (ISM) structures.

**Why this is physically motivated:**

| Structure | $\beta$ Range | Physical Basis |
| :--- | :---: | :--- |
| **Stellar Streams** | $[1.5, 2.5]$ | Streams are tidally stripped debris following near-ballistic orbits. Their surface brightness fluctuations arise from discrete stellar clumps (progenitor remnants) and stochastic stripping episodes, producing **rougher, lower-dimensional** textures with moderate spatial correlation. |
| **Galactic Cirrus** | $[2.5, 3.5]$ | Interstellar dust follows turbulent dynamics well-described by Kolmogorov theory. Observations of high-latitude cirrus (e.g., Miville-Deschênes et al. 2016) find power spectra $P(k) \propto k^{-\beta}$ with $\beta \approx 2.9$, producing **smoother, cloud-like** morphologies with strong large-scale correlation. |

**Additional realism layers:**
- **Bézier curves** approximate the great-circle-like trajectories that tidal streams follow on the sky (smooth, curvilinear arcs).
- **Gaussian smoothing** ($\sigma \in [1.5, 4.0]$ for streams, $[8.0, 15.0]$ for cirrus) controls the angular extent: streams are narrow and elongated, cirrus is diffuse and spatially extended.
- **Poisson noise injection** simulates photon-counting statistics, matching the noise regime of real CCD/CMOS detectors in survey instruments.

In [ ]:
# ==========================================
# 📐 1. FRACTAL MATH ENGINE
# ==========================================

def generate_fractal_noise(shape, exponent=3.0):
    """
    Generates 1/f^beta noise (Colored Noise) using FFT.
    
    This reproduces the power-law power spectrum P(k) ~ k^{-beta} observed
    in interstellar medium (ISM) structures. Lower beta produces rougher,
    clumpier textures (streams), while higher beta yields smoother, 
    cloud-like fields (cirrus), consistent with Kolmogorov turbulence theory.
    
    Args:
        shape: (rows, cols) of the output noise field.
        exponent: Spectral index beta. Stream: [1.5, 2.5], Cirrus: [2.5, 3.5].
    Returns:
        Normalized 2D noise field in [0, 1].
    """
    rows, cols = shape
    r, c = np.indices((rows, cols))
    center_r, center_c = rows // 2, cols // 2
    r -= center_r
    c -= center_c
    
    # Radial distance in frequency domain
    radius = np.sqrt(r**2 + c**2) + 1e-10
    
    # Power Law: 1/f^beta
    amplitude = 1.0 / (radius ** (exponent / 2.0))
    
    # Random Phase
    phase = np.random.rand(rows, cols) * 2 * np.pi
    
    # Field Construction
    field = amplitude * (np.cos(phase) + 1j * np.sin(phase))
    noise = np.fft.ifft2(np.fft.ifftshift(field)).real
    
    # Normalize [0, 1]
    return (noise - noise.min()) / (noise.max() - noise.min() + 1e-10)

def generate_procedural_structure(shape, struct_type="stream"):
    """
    Generates a synthetic astronomical structure with physically motivated morphology.
    
    - Stream: Thin, curvilinear arc (Bézier curve) with rough 1/f^beta texture 
      (beta in [1.5, 2.5]). Gaussian sigma in [1.5, 4.0] produces narrow widths 
      consistent with observed tidal tails (~0.1-0.5 deg on sky).
    - Cirrus: Diffuse, cloud-like field with smooth 1/f^beta texture (beta in [2.5, 3.5]).
      Large Gaussian sigma in [8.0, 15.0] produces extended, filamentary dust structures.
    
    Both types receive Poisson noise injection to simulate photon-counting 
    statistics inherent to CCD/CMOS astronomical detectors.
    """
    h, w = shape
    # Random Bezier Control Points
    r0, c0, r1, c1, r2, c2 = np.random.randint(0, h, 6)
    rr, cc = bezier_curve(r0, c0, r1, c1, r2, c2, weight=1)
    
    # Base Canvas
    canvas = np.zeros((h, w))
    valid = (rr < h) & (cc < w) & (rr >= 0) & (cc >= 0)
    canvas[rr[valid], cc[valid]] = 1
    
    if canvas.max() == 0: return np.zeros((h, w))
    
    # Morphological Processing
    if struct_type == "stream":
        # Stream: Defined shape, rough texture
        morphology = gaussian_filter(canvas, sigma=random.uniform(1.5, 4.0))
        beta = random.uniform(1.5, 2.5)
        texture = generate_fractal_noise((h, w), exponent=beta)
        result = morphology * (texture ** 2)
        
    else: # Cirrus
        # Cirrus: Diffuse shape, cloudy texture
        beta = random.uniform(2.5, 3.5)
        texture = generate_fractal_noise((h, w), exponent=beta)
        guide = gaussian_filter(canvas, sigma=random.uniform(8.0, 15.0))
        result = guide * texture
    
    # Poisson Noise Injection (Shot Noise Simulation)
    if result.max() > 0:
        result = result / result.max()
        # Simulate photon counts for realism
        result = np.random.poisson(result * 1000.0) / 1000.0
        
    return result

## 📡 Real Backgrounds from the Virtual Observatory

To ensure that the synthetic structures are embedded in **realistic noise environments**, backgrounds are downloaded directly from real astronomical surveys via the CDS HiPS service (Aladin Lite). Each sample is a 4-channel cube from:

| Band | Survey | Purpose |
| :--- | :--- | :--- |
| **g** (475 nm) | SDSS DR9 | Blue-optical stellar light |
| **r** (623 nm) | SDSS DR9 | Red-optical — **primary stream channel** (stream stars emit here) |
| **W1** (3.4 µm) | AllWISE | Near-IR — **primary cirrus channel** (warm dust thermal emission) |
| **W2** (4.6 µm) | AllWISE | Mid-IR — strong cirrus, minimal stellar contribution |

Coordinates are restricted to **high galactic latitude** ($b > 30°$, RA ∈ [120°, 240°], Dec ∈ [10°, 50°]) to avoid the galactic plane, where crowded star fields would dominate over LSB structures. This matches the observational strategy used by real stream surveys (e.g., the Stellar Streams Legacy Survey).

In [3]:
# ==========================================
# 📡 2. HiPS DOWNLOADER (Virtual Observatory)
# ==========================================

def download_fits_cutout(url, params):
    """Downloads a FITS cutout from the CDS Hips2Fits service."""
    try:
        # Random sleep to avoid rate limiting
        time.sleep(random.uniform(0.01, 0.1))
        r = requests.get(url, params=params, timeout=Config.TIMEOUT)
        
        if r.status_code == 200:
            with fits.open(BytesIO(r.content), ignore_missing_end=True) as hdul:
                data = hdul[0].data
                # Replace NaNs (common in survey edges) with zeros
                data = np.nan_to_num(data)
            return data
    except Exception:
        return None
    return None

def get_multiband_cube():
    """
    Attempts to download a matched 4-channel cube (g, r, W1, W2).
    Retries with new coordinates until successful.
    """
    HIPS_URL = "http://alasky.u-strasbg.fr/hips-image-services/hips2fits"
    
    while True:
        # Random Coordinates (High Latitude to avoid galactic plane clutter)
        ra = random.uniform(120, 240)
        dec = random.uniform(10, 50)
        
        layers = []
        failed = False
        
        for survey in Config.SURVEYS:
            params = {
                'hips': survey,
                'width': Config.IMG_SIZE,
                'height': Config.IMG_SIZE,
                'fov': 0.1, # Field of View (degrees)
                'projection': 'TAN',
                'coordsys': 'icrs',
                'ra': ra,
                'dec': dec,
                'format': 'fits'
            }
            
            data = download_fits_cutout(HIPS_URL, params)
            
            # Validity check: Reject empty or corrupted images
            if data is None or np.std(data) < 0.00001:
                failed = True
                break
            layers.append(data)
            
        if not failed and len(layers) == 4:
            return np.array(layers)

## 🧬 Physics-Informed Injection Pipeline

The injection step is where astrophysical realism is enforced at the **spectral level**. Rather than injecting the same signal into all bands, the pipeline applies physically motivated amplitude ratios:

**Stellar Streams** (old stellar populations):
- Dominant in **g** (×0.8) and **r** (×1.0) — stars emit thermal radiation peaking in the optical.
- Negligible in W1/W2 — resolved/unresolved stellar populations have minimal mid-IR emission.

**Galactic Cirrus** (interstellar dust):
- Dominant in **W1** (×2.0) and **W2** (×3.0) — dust grains absorb UV/optical starlight and re-emit thermally at $T \sim 15\text{–}30\,\mathrm{K}$, peaking in the far-IR but still significant at 3–5 µm.
- Weak leakage in **g** (×0.4) and **r** (×0.5) — optical scattering by dust grains (reflection nebulae effect).

**Signal intensity** is calibrated relative to the **local background noise floor** (`np.std` per band), producing realistic Signal-to-Noise Ratios rather than arbitrary pixel values.

### Hard Mining: The "Crossing Problem"
In hard mode, the generator **forces co-occurrence** (90% probability for both structures simultaneously) with:
- **Very bright cirrus** (SNR ×4–9) masking **very faint streams** (SNR ×0.7–2.5).
- This mimics the real-world scenario where cirrus contamination is the primary source of false positives in stream surveys.

In [4]:
# ==========================================
# 🧬 3. INJECTION PIPELINE (NORMAL + HARD)
# ==========================================

def process_single_sample(idx, mode="normal"):
    """
    Main pipeline step: 
    1. Gets real background.
    2. Injects synthetic structures based on difficulty mode.
    3. Saves FITS and Mask.
    """
    try:
        # A. Get Background
        cube = get_multiband_cube()
        if cube is None: return False
        
        _, h, w = cube.shape
        final_cube = cube.copy()
        final_mask = np.zeros((h, w), dtype=np.uint8)
        
        # Estimate background noise levels per band
        noise_r = np.std(cube[1])  # r-band (Stream target)
        noise_w1 = np.std(cube[2]) # W1-band (Cirrus target)
        
        # B. Parameter Setup
        if mode == "normal":
            # Independent probabilities
            prob_cirrus = 0.8
            prob_stream = 0.5
            # Standard intensities (Signal-to-Noise Ratio)
            intensity_cirrus = (2.0, 5.0) 
            intensity_stream = (1.5, 4.0)
            
        else: # HARD MODE (Hard Mining)
            # Forced Co-occurrence (The "Crossing" Problem)
            prob_cirrus = 0.9 
            prob_stream = 0.9 
            # High Contrast / Low Signal
            intensity_cirrus = (4.0, 9.0)  # Very bright cirrus
            intensity_stream = (0.7, 2.5)  # Very faint stream (Critical SNR)

        # C. Injection
        
        # 1. Inject Cirrus (Class 2)
        if random.random() < prob_cirrus:
            cirrus_map = generate_procedural_structure((h, w), "cirrus")
            
            # Base intensity scaled by local noise floor
            amp = max(noise_w1, 0.01) * random.uniform(*intensity_cirrus)
            
            # Inject mostly in IR bands (Physics-Informed)
            final_cube[2] += cirrus_map * amp * 2.0 # W1
            final_cube[3] += cirrus_map * amp * 3.0 # W2
            # Some leakage into visible
            final_cube[0] += cirrus_map * amp * 0.4 # g
            final_cube[1] += cirrus_map * amp * 0.5 # r
            
            # Update Mask
            final_mask[cirrus_map > cirrus_map.max() * 0.15] = 2
            
        # 2. Inject Stream (Class 1) - Overwrites Cirrus in Mask
        if random.random() < prob_stream:
            stream_map = generate_procedural_structure((h, w), "stream")
            
            amp = max(noise_r, 0.01) * random.uniform(*intensity_stream)
            
            # Hard Mode Boost: If Cirrus is present, boost Stream slightly 
            # to ensure physical detectability is possible
            if mode == "hard" and final_mask.max() == 2:
                amp *= 1.2
                
            # Inject mostly in Visible bands (Stellar population)
            final_cube[0] += stream_map * amp * 0.8 # g
            final_cube[1] += stream_map * amp * 1.0 # r
            
            # Update Mask (Priority 1)
            final_mask[stream_map > stream_map.max() * 0.2] = 1

        # D. Save to Disk
        # We use a single running index for filenames
        filename = f"img_{mode}_{idx:05d}.fits"
        maskname = f"mask_{mode}_{idx:05d}.fits"
        
        fits.writeto(os.path.join(Config.IMG_PATH, filename), final_cube, overwrite=True)
        fits.writeto(os.path.join(Config.MASK_PATH, maskname), final_mask, overwrite=True)
        
        return True
        
    except Exception:
        return False

# ==========================================
# 🚀 4. EXECUTION ENGINE
# ==========================================

print(f"🔥 Starting Generation on {Config.MAX_WORKERS} threads...")

with ThreadPoolExecutor(max_workers=Config.MAX_WORKERS) as executor:
    
    # Phase 1: Normal Dataset
    print(f"📦 Generating {Config.N_NORMAL} Normal samples...")
    # Map index 0 to N_NORMAL
    list(tqdm(executor.map(lambda i: process_single_sample(i, "normal"), range(Config.N_NORMAL)), total=Config.N_NORMAL))
    
    # Phase 2: Hard Dataset
    print(f"😈 Generating {Config.N_HARD} Hard Mining samples...")
    # Map index N_NORMAL to TOTAL
    list(tqdm(executor.map(lambda i: process_single_sample(i, "hard"), range(Config.N_NORMAL, Config.TOTAL)), total=Config.N_HARD))

print("\n✅ Dataset Generation Complete.")
print(f"📁 Output Location: {Config.BASE_OUTPUT}")

🔥 Starting Generation on 8 threads...
📦 Generating 12000 Normal samples...


  0%|          | 0/12000 [00:00<?, ?it/s]

😈 Generating 8000 Hard Mining samples...


  0%|          | 0/8000 [00:00<?, ?it/s]


✅ Dataset Generation Complete.
📁 Output Location: /kaggle/working/synthetic_dataset
